# Synthetic Data Exploration & Model Training
This notebook explores the generated synthetic ledger and trains an XGBoost classifier to predict liquidity crunches.

In [ ]:
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import shap

%matplotlib inline

## 1. Load Data
We load the `ml_feature_store.csv` generated by `data/generator/generate_synthetic_ledger.py`.

In [ ]:
# Ensure you have run generate_synthetic_ledger.py first
df = pd.read_csv('../data/generator/ml_feature_store.csv')
df.head()

In [ ]:
# Basic EDA
sns.countplot(x='is_liquidity_crunch', data=df)
plt.title('Distribution of Liquidity Crunch')
plt.show()

## 2. Train XGBoost Model
We train a model to predict `is_liquidity_crunch`.

In [ ]:
features = [
    'avg_monthly_inflow', 'income_entropy', 'burn_rate_first_10d',
    'rolling_outflow_mean_7d', 'rolling_outflow_std_7d', 'velocity_acceleration',
    'cash_out_dependency_ratio', 'avoidable_fee_leakage', 'fixed_obligation_ratio',
    'discretionary_spend_ratio'
]
X = df[features]
y = df['is_liquidity_crunch']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

## 3. Explainability with SHAP
Let's extract SHAP values to explain what drives a liquidity crunch.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)